# Private HASS SFT: capture to adapter
Run one section at a time from the updated `nlp-lab` checkout in Jupyter. Prerequisites: Dagster deployed from `feature/gold-materialization`; `eliza_hass`, `eliza_training` and `eliza_storage` from the current `eliza-spark` Jupyter helpers; private MinIO access and DGX control configured through the existing secret mounts. Keep this notebook and its outputs private.
In the notebook kernel, install the updated lightweight client with `%pip install -e /home/jovyan/work/nlp-lab/packages/training-client` and restart the kernel if it still uses the older image client. CUDA/training packages are prepared on the DGX inside the Slurm allocation.
**Start here for a fresh run:** create and upload a full Assist snapshot using the recipe below. Set `CAPTURE_FILE` and `HOUSEHOLD_UUID` in the setup cell. Keep `slurm-artifacts` if that is your private run bucket. Run the sections in order. The pipeline proposes the split and computes the dataset pins. Use a new `PRIVATE_DIR` for a different capture or experiment.
The [runbook](../docs/runs/hass-private-sft-smoke.md) covers recovery and acceptance.

In [ ]:
from pathlib import Path, PurePosixPath
import hashlib, inspect, json, re, uuid
import yaml
from eliza_storage import ElizaStorage
from eliza_hass import HassPilotWorkflow, HassSftWorkflow
from eliza_training import ElizaTraining

REPO = Path('/home/jovyan/eliza/nlp-lab')
PRIVATE_DIR = Path('/home/jovyan/eliza/hass-private-smoke-01')  # retain this across restarts
SLURM_ARTIFACT_BUCKET = 'slurm-artifacts'  # configured private writable training bucket
SOURCE_BUCKET = 'synthetic'
DESTINATION_BUCKET = 'sft-datasets'
CAPTURE_FILE = '/home/jovyan/eliza/datasets/hass-snapshots/d30157dc-5a19-4dc3-8f01-3dc86caee73d.json'  # path to the full JSON uploaded in JupyterLab
HOUSEHOLD_UUID = 'd40134f7-64b7-40eb-bc70-a04e9c4bafcc'  # existing stable UUID for this home (reuse across captures)

LIMIT = 32 * 1024 * 1024
storage = ElizaStorage()
def sha(data):
    return hashlib.sha256(data).hexdigest()
if 'restricted_manifest_ack_sha256' not in inspect.signature(ElizaTraining.submit).parameters:
    raise RuntimeError('Update the editable nlp-lab training client and restart the kernel')
if not (REPO / 'configs/sft_hass_private_smoke.template.yaml').is_file():
    raise RuntimeError('Update the nlp-lab checkout or correct REPO')
PRIVATE_DIR.mkdir(mode=0o700, parents=True, exist_ok=True)
progress_file = PRIVATE_DIR / 'pipeline-progress.json'
progress = json.loads(progress_file.read_text()) if progress_file.exists() else {}
def remember(**values):
    if any(key in progress and progress[key] != value for key, value in values.items()):
        raise ValueError('This directory is bound to different pipeline inputs; use a new PRIVATE_DIR')
    progress.update(values)
    temporary = progress_file.with_suffix('.tmp')
    temporary.write_text(json.dumps(progress, indent=2))
    temporary.replace(progress_file)


## 1. Capture, archive, ingest and generate
With the [Assist Snapshot integration](https://github.com/farkas93/eliza-data-pipelines/blob/feature/gold-materialization/docs/assist-snapshot-export.md) installed, run this in **Home Assistant → Developer Tools → Actions** (YAML mode):
```yaml
action: assist_snapshot.capture
data:
  mode: full
  api_ids: [assist]
  platform: llama_conversation
  allow_runtime_side_effects: true
  inventory_scope: exposed
  save: true
```
Download the saved `/config/assist_snapshots/<capture-uuid>.json` from HA, upload it through the JupyterLab file browser, and set `CAPTURE_FILE` above. A partial capture cannot generate this pilot. If you have never assigned a household UUID, generate one with `print(uuid.uuid4())` and retain it for this household.
On restart, rerun setup and these cells with the same `PRIVATE_DIR`. The saved archive key and Dagster run IDs reconnect to the same runs.

In [ ]:
if not HOUSEHOLD_UUID or not (CAPTURE_FILE or progress.get('source_key')):
    raise ValueError('Set HOUSEHOLD_UUID and the uploaded CAPTURE_FILE in the setup cell')
remember(household_uuid=HOUSEHOLD_UUID, source_bucket=SOURCE_BUCKET)
pilot = HassPilotWorkflow(source_id=HOUSEHOLD_UUID, bucket=SOURCE_BUCKET, storage=storage)
pilot.check_jobs()
if progress.get('source_key'):
    pilot.use_archived_source(progress['source_key'])
if CAPTURE_FILE:
    pilot.archive_file(CAPTURE_FILE)
remember(source_key=pilot.source_key)
pilot.ingest_run_id = progress.get('ingest_run_id')
INGEST_RUN_ID = pilot.launch_ingestion()
remember(ingest_run_id=INGEST_RUN_ID)
print(pilot.source_key, INGEST_RUN_ID)
pilot.watch('ingest')


In [ ]:
pilot.generate_run_id = progress.get('generate_run_id')
GENERATE_RUN_ID = pilot.launch_generation(seed=42, name_variant='original', max_model_rounds=3)
remember(generate_run_id=GENERATE_RUN_ID)
print(GENERATE_RUN_ID)
pilot.watch('generate')


In [ ]:
CANDIDATES_KEY = pilot.preview(max_episodes=1)['key']
CANDIDATES_SHA256 = sha(storage.read_bytes(bucket=SOURCE_BUCKET, key=CANDIDATES_KEY, max_bytes=LIMIT))
remember(candidates_key=CANDIDATES_KEY, candidates_sha256=CANDIDATES_SHA256)
print(CANDIDATES_KEY, CANDIDATES_SHA256)


## 2. Preview and freeze the smoke dataset
The preview job proposes the closest feasible split to 20% held-out episodes. It ranks whole related-task groups by a stable policy hash to break ties and requires eval to include both a tool call and a post-tool reply. This measures holdout within one home. The 24-episode pilot has 49 assistant-decision rows; an episode contributes all of its rows to one split. No assignments to enter. Preview shows the exact split counts, held-out groups and examples before you freeze. Candidates remain canonically unreviewed and council is not run.

In [ ]:
sft = HassSftWorkflow(candidates_key=CANDIDATES_KEY, candidates_sha256=CANDIDATES_SHA256,
                      bucket=SOURCE_BUCKET, storage=storage)
if not hasattr(sft, 'launch_preview'):
    raise RuntimeError('Update eliza_hass.py from the current Jupyter helpers before previewing')
sft.check_jobs(automatic=True)
sft.preview_run_id = progress.get('preview_run_id')
remember(preview_run_id=sft.launch_preview())
sft.watch('preview')
preview = sft.resolve_preview()
for split, counts in preview['splits'].items():
    print(f'{split}: {counts["episodes"]} episodes, {counts["rows"]} SFT rows, {counts["groups"]} task groups')
print('Excluded episodes:', preview['excluded_episodes'])
for group in preview['groups']:
    if group['split'] == 'eval':
        print('Held-out group', group['base_task_id'], ', '.join(group['families']),
              f'({len(group["episode_ids"])} episodes, {group["rows"]} rows)', ', '.join(group['coverage']))


In [ ]:
def show_example(split, kind, *, full=False):
    row = preview['examples'][split][kind]
    print(f'{split} / {kind} / {row["family"]}')
    if full:
        print('Advertised schemas:', row['tools_json'])
    else:
        print('System prompt and tool schemas omitted here (full=True to show)')
    for message in row['messages'][0 if full else 1:]:
        print(message['role'] + ':', message.get('content'), message.get('tool_calls', ''))
    print('Exact target:', json.dumps(row['target_message'], ensure_ascii=False, indent=2))
for split, examples in preview['examples'].items():
    for kind in examples:
        show_example(split, kind)


In [ ]:
# Run this cell to freeze the exact split shown above.
sft.selection_run_id = progress.get('automatic_selection_run_id')
SELECTION_RUN_ID = sft.launch_automatic_selection(expected_selection_id=preview['selection_id'])
remember(automatic_selection_run_id=SELECTION_RUN_ID)
print(SELECTION_RUN_ID)
sft.watch('select')
selection = sft.resolve_selection()
remember(selection_sha256=selection['selection_sha256'])
print('selection SHA-256:', selection['selection_sha256'])


In [ ]:
sft.export_run_id = progress.get('automatic_export_run_id')
EXPORT_RUN_ID = sft.launch_export(destination_bucket=DESTINATION_BUCKET)
remember(automatic_export_run_id=EXPORT_RUN_ID)
print(EXPORT_RUN_ID)
sft.watch('export')
receipt = sft.resolve_export()
receipt['data']


## 3. Prepare, submit and monitor the 12-step run
The next cells populate the private config from the verified export, preserve a run UUID in `PRIVATE_DIR`, and require your exact P4 manifest acknowledgement before switching DGX modes. Three different hashes appear in this flow: the **candidate** SHA-256 pins the generated episodes, the **selection** SHA-256 pins the frozen train/eval split, and the **manifest** SHA-256 pins the exported Parquet dataset. The submission prompt asks for the **manifest** SHA-256 (`receipt['manifest_sha256']`), because that is the dataset the trainer will actually read. The trainer verifies all selected rows with the pinned tokenizer before loading weights. The submission cell is one-shot for this directory. After a lost response, use the recovery cell and keep the same UUID; do not automatically submit again. On reconnect, reuse the same directory and Dagster IDs.

In [ ]:
if not SLURM_ARTIFACT_BUCKET:
    raise ValueError('Set SLURM_ARTIFACT_BUCKET to the configured private training bucket')
pin = {'uri': receipt['manifest_uri'], 'sha256': receipt['manifest_sha256']}
cfg = yaml.safe_load((REPO / 'configs/sft_hass_private_smoke.template.yaml').read_text())
cfg['data'].update(bucket=receipt['bucket'], dataset_id=receipt['dataset_id'],
                   dataset_version=receipt['dataset_version'], expected_manifest_sha256=pin['sha256'])
journal = PRIVATE_DIR / 'run.json'
binding = {'dataset_manifest': pin, 'SLURM_ARTIFACT_BUCKET': SLURM_ARTIFACT_BUCKET}
if not journal.exists():
    with journal.open('x') as out:
        json.dump({**binding, 'run_id': str(uuid.uuid4())}, out)
saved = json.loads(journal.read_text())
if any(saved.get(key) != value for key, value in binding.items()):
    raise ValueError('This run directory belongs to another dataset or artifact bucket')
run_id = str(uuid.UUID(saved['run_id']))
config_path = REPO / f'configs/private-hass-{run_id}.yaml'
config_text = yaml.safe_dump(cfg, sort_keys=False)
if config_path.exists() and config_path.read_text() != config_text:
    raise ValueError('Existing run config differs; inspect before proceeding')
if not config_path.exists():
    with config_path.open('x') as out:
        out.write(config_text)
training = ElizaTraining(bucket=SLURM_ARTIFACT_BUCKET, storage=storage)
print('run_id:            ', run_id)
print('dataset manifest:  ', pin['uri'])
print('manifest SHA-256:  ', pin['sha256'], '<- type this value at the submit prompt')
print('selection SHA-256: ', progress.get('selection_sha256'), '<- NOT this; that is the frozen split record')


In [ ]:
attempt = PRIVATE_DIR / 'submit-attempt.json'
if attempt.exists():
    raise RuntimeError('This run already attempted submission; inspect status or use recovery')
print('Dataset manifest:', pin['uri'])
print('Enter this exact manifest SHA-256 below:', pin['sha256'])
ack = input('Manifest SHA-256 (not the selection SHA-256): ').strip()
if ack != pin['sha256']:
    raise ValueError('Acknowledgement does not match the frozen manifest')
training.environment('training')
with attempt.open('x') as out:
    json.dump({'run_id': run_id}, out)
training.submit(repo=REPO, config=str(config_path.relative_to(REPO)), run_id=run_id,
                dataset_manifest=pin, restricted_manifest_ack_sha256=ack,
                runtime_profile='trl', runtime_source=REPO / 'packages/training-client',
                cpus=8, memory_mb=32768, time_minutes=90)


In [ ]:
# Run only if the submission response was lost and the receipt is absent.
# training.discover()
# training.recover_submission(run_id)


In [ ]:
if training.environment().get('phase') == 'training':
    training.discover()  # also needed after a kernel reconnect
training.watch(run_id, timeout_seconds=7200, on_update=print)


## 4. Verify and retrieve the adapter
Check the worker result and optimizer step count, then retrieve final adapter/tokenizer files and verify each against the archived index. Slurm completion alone is not sufficient. If the worker result is absent, inspect Slurm stdout/stderr and the archived training log.

In [ ]:
prefix = f'{training.prefix}/{run_id}'
def run_json(name):
    return storage.read_json(bucket=SLURM_ARTIFACT_BUCKET, key=f'{prefix}/{name}', max_bytes=LIMIT)
result = run_json('result.json')
log = training.logs(run_id, max_bytes=LIMIT)
if result.get('run_id') != run_id or result.get('outcome') != 'succeeded':
    raise RuntimeError(f'Worker acceptance failed: {result}')
steps = cfg['training']['max_steps']
if not re.search(rf'Training finished at optimizer global_step={steps}(?:\s|$)', log):
    raise RuntimeError('No optimizer step completion found in the archived training log')
index = run_json('artifact-index.json')
final_entries = [entry for entry in index if PurePosixPath(entry['path']).parent.name == 'final']
names = {PurePosixPath(entry['path']).name for entry in final_entries}
if not {'adapter_config.json', 'adapter_model.safetensors', 'tokenizer.json'} <= names or len(names) != len(final_entries):
    raise ValueError('Expected one complete final adapter/tokenizer directory')
final_dir = PRIVATE_DIR / 'final'
final_dir.mkdir(mode=0o700, exist_ok=True)
for entry in final_entries:
    rel = PurePosixPath(entry['path'])
    if rel.is_absolute() or '..' in rel.parts or not 0 < entry['size'] <= 128 * 1024 * 1024:
        raise ValueError('Unexpected artifact path or size')
    data = storage.read_bytes(bucket=SLURM_ARTIFACT_BUCKET, key=f'{prefix}/artifacts/{rel}', max_bytes=entry['size'])
    if len(data) != entry['size'] or sha(data) != entry['sha256']:
        raise ValueError(f'Artifact checksum mismatch: {rel.name}')
    dest = final_dir / rel.name
    if dest.exists() and dest.read_bytes() != data:
        raise ValueError(f'Existing local file differs: {dest.name}')
    if not dest.exists():
        with dest.open('xb') as out:
            out.write(data)
adapter = json.loads((final_dir / 'adapter_config.json').read_text())
model_id = f"{cfg['model']['owner']}/{cfg['model']['name']}"
if adapter.get('base_model_name_or_path') != model_id or adapter.get('revision') != cfg['model']['revision']:
    raise ValueError('Adapter base lineage does not match the pinned config')
print('Verified', steps, 'steps and final adapter files:', sorted(names))


## 5. Restore inference
Run explicitly once Slurm jobs finish. The controller refuses a busy switch. This comparison uses a separate CPU environment, so the DGX need not remain in training mode. If training fails, inspect or explicitly cancel the active job before restoring.

In [ ]:
if input('Type RESTORE to return the DGX to inference: ').strip() != 'RESTORE':
    raise RuntimeError('Inference restoration not requested')
training.environment('inference')


## 6. Reload the adapter and compare held-out examples
Run these cells in a kernel with the `nlp-lab` checkout, CPU PyTorch, Transformers 5.7.0, PEFT, Datasets 4.8.5 and Jinja2. The Jupyter image’s lightweight client does not supply those packages. Allow memory for the 0.8B model and time for CPU generation. If changing kernels, rerun the setup/reconnect and artifact verification cells first. This step decodes locally: it never executes generated tool calls. Compare output to the frozen target; the smoke is not an improvement benchmark.

In [ ]:
import sys
sys.path.insert(0, str(REPO))
import torch
from peft import PeftModel
from transformers import AutoTokenizer
from transformers import Qwen3_5ForConditionalGeneration
from src.sft_dataset_loader import load_sft_manifest_dataset, tokenize_with_loss_mode

mirror = PRIVATE_DIR / 'dataset-mirror'
manifest_key = receipt['manifest_key']
manifest_bytes = storage.read_bytes(bucket=receipt['bucket'], key=manifest_key, max_bytes=LIMIT)
if sha(manifest_bytes) != pin['sha256']:
    raise ValueError('Frozen dataset manifest bytes changed')
manifest = json.loads(manifest_bytes)
mirror.mkdir(mode=0o700, exist_ok=True)
for split in manifest['splits']:
    split_name = split['split']
    if split_name not in ('train', 'eval'):
        raise ValueError('Unexpected split')
    part = mirror / f'split={split_name}/part-00000.parquet'
    part.parent.mkdir(mode=0o700, exist_ok=True)
    payload = storage.read_bytes(bucket=receipt['bucket'], key=split['key'], max_bytes=LIMIT)
    if sha(payload) != split['sha256']:
        raise ValueError('Frozen dataset split bytes changed')
    part.write_bytes(payload)
(mirror / 'manifest.json').write_bytes(manifest_bytes)
held = load_sft_manifest_dataset(manifest_uri=str(mirror / 'manifest.json'), train_split='train', eval_split='eval',
                                 max_train_samples=None, max_eval_samples=None, expected_manifest_sha256=pin['sha256']).eval_dataset
ordered = sorted(range(len(held)), key=lambda i: held[i]['example_id'])
chosen = [next((i for i in ordered if held[i]['target_message']['tool_calls']), None),
          next((i for i in ordered if held[i]['target_message']['content'] and held[i]['messages'][-1]['role'] == 'tool'), None)]
if None in chosen:
    raise ValueError('Held-out split needs a tool call and a post-tool speech example')
fixed = held.select(chosen)
tokenizer = AutoTokenizer.from_pretrained(str(final_dir), local_files_only=True)
encoded = tokenize_with_loss_mode(fixed, tokenizer, cfg['model']['max_seq_len'],
                                  loss_mode='assistant_only', tokenizer_type='qwen', enable_thinking=False)
base = Qwen3_5ForConditionalGeneration.from_pretrained(model_id, revision=cfg['model']['revision'], dtype=torch.float32)
model = PeftModel.from_pretrained(base, str(final_dir)).eval()
comparison = []
for example, row in zip(fixed, encoded, strict=True):
    boundary = next(i for i, label in enumerate(row['labels']) if label != -100)
    prompt = torch.tensor([row['input_ids'][:boundary]])
    args = dict(input_ids=prompt, attention_mask=torch.ones_like(prompt), do_sample=False, max_new_tokens=96)
    with torch.inference_mode():
        with model.disable_adapter():
            original = model.generate(**args)[0, boundary:]
        tuned = model.generate(**args)[0, boundary:]
    comparison.append({'example_id': example['example_id'], 'target': tokenizer.decode(row['input_ids'][boundary:]),
                       'base': tokenizer.decode(original), 'adapter': tokenizer.decode(tuned)})
(PRIVATE_DIR / 'held-out-comparison.json').write_text(json.dumps(comparison, indent=2, ensure_ascii=False))
comparison
